In [ ]:

import torch
import os, random
import numpy as np
from transformers import set_seed

SEED = 159753


# 1) Python, NumPy, PyTorch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 2) HF helper
set_seed(SEED)

In [ ]:
model_path = "google/siglip-base-patch16-224"
device = "cuda"


In [ ]:
test_dataset = "/run/media/victor/pessoal/mestrado/codigo/datasets/v6_test_leiomyoma.csv"


In [ ]:
torch.set_float32_matmul_precision('high')


In [ ]:
from seq_aux_dataloader_aug import siglipFinetuner
from transformers import SiglipProcessor, SiglipModel
import torch


def load_model():
    trained_model = SiglipModel.from_pretrained(
        model_path,
        device_map=device,
    )
    trained_model = torch.compile(trained_model)
    processor = SiglipProcessor.from_pretrained(model_path)
    trained_model = siglipFinetuner(trained_model, processor, lr = 0.000005, only_projection=True)
    return trained_model, processor


In [ ]:
import gc
gc.collect()


In [ ]:
with torch.no_grad():
    torch.cuda.empty_cache()


In [ ]:
import pandas as pd

test_df = pd.read_csv(test_dataset)
test_df = test_df[test_df["image_path"].str.startswith("/run/media/victor/pessoal/mestrado/dataset/leiomioma/")]
print(f"Total validation images: {len(test_df)}")


In [ ]:
from tqdm import tqdm
import numpy as np
from PIL import Image


def process_image_patches_optimized(image_path, model, processor, patch_size=224, step=15, 
                                  text_query="vessel", threshold=0.1, batch_size=8):

    image = np.asarray(Image.open(image_path).convert("RGB"))
    h, w, _ = image.shape
    sim_image = np.zeros((h, w), dtype=np.float32)
    
    text_inputs = processor(text=[text_query], return_tensors="pt", padding="max_length")
    text_inputs = {k: v.to("cuda") for k, v in text_inputs.items()}
    
    patch_coords = []
    for i in range(0, h - patch_size + 1, step):
        for j in range(0, w - patch_size + 1, step):
            patch_coords.append((i, j))
    
    patches = []
    model.eval()
    
    with torch.no_grad():
        for batch_start in range(0, len(patch_coords), batch_size):
            batch_end = min(batch_start + batch_size, len(patch_coords))
            batch_coords = patch_coords[batch_start:batch_end]
            
            batch_patches = []
            for i, j in batch_coords:
                patch = image[i:i + patch_size, j:j + patch_size]
                batch_patches.append(patch)
            
            if batch_patches:
                pil_patches = [Image.fromarray(patch) for patch in batch_patches]
                batch_inputs = processor(images=pil_patches, return_tensors="pt", padding="max_length")
                batch_inputs = {k: v.to("cuda") for k, v in batch_inputs.items()}
                
                outputs = model(**batch_inputs, **text_inputs)
                similarities = torch.sigmoid(outputs.logits_per_image).cpu().numpy().flatten()
                
                for idx, (i, j) in enumerate(batch_coords):
                    sim_value = similarities[idx]
                    
                    if sim_value > threshold:
                        patches.append(batch_patches[idx])
                    
                    sim_image[i:i + patch_size, j:j + patch_size] = np.maximum(
                        sim_image[i:i + patch_size, j:j + patch_size], 
                        sim_value
                    )
    
    return patches, sim_image


In [ ]:
model_files = ["/run/media/victor/pessoal/mestrado/codigo/train/checkpoint/V6_phase_2/siglip-epoch4-val_loss4.12122-batch200-lr2.50e-05-v1931-seed1561312-modelo_basesiglip-epoch1-val_loss2.78238-batch100-lr5.00e-05-v1347-seed1561312-modelo_basephase_1.ckpt"]

In [ ]:
model_files

In [ ]:
queries = [
    "compaction of the stroma",
    "fibroplasia",
    "vessel"
]

all_model_results = {}


In [ ]:
import os

for model_idx, model_file in enumerate(model_files, 1):
    model_name = os.path.basename(model_file)
    print(f"\n{'='*80}")
    print(f"Evaluating Model {model_idx}/{len(model_files)}: {model_name}")
    print(f"{'='*80}")
    
    print("Loading model...")
    trained_model, processor = load_model()
    checkpoint = torch.load(model_file)
    trained_model.load_state_dict(checkpoint["state_dict"])
    trained_model = trained_model.siglip_model
    trained_model.eval()
    print("Model loaded!")
    
    model_df = test_df.copy()
    imgs = model_df["image_path"].to_list()

    for path in tqdm(imgs, desc=f"Processing images"):
        for query in queries:
            if "vessel" in query:
                patch_size = 112
            else:
                patch_size = 224
            
            patches, sim_image = process_image_patches_optimized(
                path, 
                trained_model, 
                processor, 
                patch_size=patch_size, 
                step=14, 
                text_query=query, 
                threshold=0.2, 
                batch_size=100  
            )
            
            model_df.loc[model_df["image_path"] == path, query] = max(sim_image.flatten())

    del trained_model, processor, checkpoint
    torch.cuda.empty_cache()
    gc.collect()
    
    output_csv = f"best_siglip_test_similarity_model_{model_idx}.csv"
    model_df.to_csv(output_csv, index=False)
    all_model_results[model_name] = model_df
    print(f"Saved results to {output_csv}")
    
print("\n" + "="*80)
print("All models evaluated!")
print("="*80)


In [ ]:
sim_image

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, f1_score, matthews_corrcoef, balanced_accuracy_score

vessel_regex = r"\bvessel\b|\bvenule\b|\barteriole\b|\bvein\b|\bvessels\b|\bvenules\b|\barterioles\b|\bveins\b|\bvascularization\b|\bvascularity\b|\bvascular\b|\bvascularized\b"
stroma_regex = r"\bcompaction of the stroma\b|\bcompacted stroma\b|\bcollagen compaction\b|\bstroma is compacted\b|\bstromal compaction\b"


def evaluate_model_performance(df, model_name):
    df["caption"] = df["caption"].str.lower()
    df['has_fibroplasia'] = df["caption"].str.contains("fibroplasia").astype(int)
    df['has_vessel'] = df["caption"].str.contains(vessel_regex, case=False, na=False, regex=True).astype(int)
    df['has_stroma'] = df["caption"].str.contains(stroma_regex, case=False, na=False, regex=True).astype(int)
    
    results = {}
    
    for condition_name, score_column, label_column in [
        ('fibroplasia', 'fibroplasia', 'has_fibroplasia'),
        ('vessel', 'vessel', 'has_vessel'),
        ('stroma', 'compaction of the stroma', 'has_stroma')
    ]:
        if df[label_column].sum() == 0:
            continue
        
        test_thresholds = np.linspace(df[score_column].min(), df[score_column].max(), 1000)
        balanced_acc_scores = []
        accuracy_scores = []
        
        for thresh in test_thresholds:
            predictions = (df[score_column] >= thresh).astype(int)
            balanced_acc_scores.append(balanced_accuracy_score(df[label_column], predictions))
            accuracy_scores.append((predictions == df[label_column]).mean())
        
        # Find optimal threshold
        bal_acc_max_idx = np.argmax(balanced_acc_scores)
        
        # Calculate confusion matrix values at optimal Balanced Accuracy threshold
        optimal_thresh_bal_acc = test_thresholds[bal_acc_max_idx]
        predictions_at_bal_acc = (df[score_column] >= optimal_thresh_bal_acc).astype(int)
        
        tp_bal_acc = ((predictions_at_bal_acc == 1) & (df[label_column] == 1)).sum()
        tn_bal_acc = ((predictions_at_bal_acc == 0) & (df[label_column] == 0)).sum()
        fp_bal_acc = ((predictions_at_bal_acc == 1) & (df[label_column] == 0)).sum()
        fn_bal_acc = ((predictions_at_bal_acc == 0) & (df[label_column] == 1)).sum()
        val_accuracy_bal_acc = (predictions_at_bal_acc == df[label_column]).mean()
        
        results[condition_name] = {
            'max_balanced_acc': max(balanced_acc_scores),
            'optimal_threshold_balanced_acc': optimal_thresh_bal_acc,
   
            'tp_bal_acc': int(tp_bal_acc),
            'tn_bal_acc': int(tn_bal_acc),
            'fp_bal_acc': int(fp_bal_acc),
            'fn_bal_acc': int(fn_bal_acc),
            'val_accuracy_bal_acc': float(val_accuracy_bal_acc),
  
            'max_accuracy': max(accuracy_scores)
        }

    return results


model_performance = {}

for model_name, df in all_model_results.items():
    print(f"\nEvaluating: {model_name}")
    performance = evaluate_model_performance(df, model_name)
    model_performance[model_name] = performance



In [ ]:
# all_model_results["siglip-epoch4-val_loss4.12122-batch200-lr2.50e-05-v1931-seed1561312-modelo_basesiglip-epoch1-val_loss2.78238-batch100-lr5.00e-05-v1347-seed1561312-modelo_basephase_1.ckpt"]

In [ ]:
siglip_results = pd.DataFrame(performance)

In [ ]:
siglip_results

In [ ]:
siglip_results.to_csv("/run/media/victor/pessoal/mestrado/codigo/comparison/v2_siglip_results.csv", index=False)

In [ ]:
pd.read_csv("/run/media/victor/pessoal/mestrado/codigo/comparison/siglip_results.csv")